# Healthcare AI — Updated Datasets & Models

This notebook documents the upgraded datasets (UCI sources), EDA, and retrained models.


## 1) Setup

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

ROOT = Path('.').resolve()
sns.set_theme(style='whitegrid')


## 2) Dataset sources

In [ ]:
print((ROOT/'data'/'SOURCES.md').read_text(encoding='utf-8'))


## 3) Load datasets

In [ ]:
diabetes = pd.read_csv(ROOT/'data'/'diabetes.csv')
heart = pd.read_csv(ROOT/'data'/'heart.csv')
cancer = pd.read_csv(ROOT/'data'/'breast_cancer_wdbc.csv')

display(diabetes.head())
display(heart.head())
display(cancer.head())

print('Shapes:')
print('diabetes', diabetes.shape)
print('heart   ', heart.shape)
print('cancer  ', cancer.shape)


## 4) Quick EDA (missingness, target balance)

In [ ]:
def quick_eda(df, target='target'):
    print('Missing values:', int(df.isna().sum().sum()))
    print('Target balance:')
    print(df[target].value_counts(normalize=True).rename('share'))

print('--- Diabetes ---')
quick_eda(diabetes)
print('\n--- Heart ---')
quick_eda(heart)
print('\n--- Cancer ---')
quick_eda(cancer)


## 5) Load trained pipelines + metrics

In [ ]:
import pickle

def load_artifacts(key):
    pipe = pickle.load(open(ROOT/'model'/f'{key}_pipeline.pkl','rb'))
    metrics = json.loads((ROOT/'model'/f'{key}_metrics.json').read_text(encoding='utf-8'))
    return pipe, metrics

pipes = {}
metrics = {}
for k in ['diabetes','heart','cancer']:
    pipes[k], metrics[k] = load_artifacts(k)
    print(k, metrics[k]['best_model'], metrics[k]['test_metrics'])


## 6) ROC curves (sanity-check on holdout split used in training script)

In [ ]:
from sklearn.model_selection import train_test_split

datasets = {'diabetes': diabetes, 'heart': heart, 'cancer': cancer}

plt.figure(figsize=(8,6))
for k, df in datasets.items():
    y = df['target'].astype(int).to_numpy()
    X = df.drop(columns=['target'], errors='ignore')
    if k == 'cancer' and 'id' in X.columns:
        X = X.drop(columns=['id'])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    prob = pipes[k].predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{k} (AUC={auc:.3f})")
plt.plot([0,1],[0,1],'k--',alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (Holdout)')
plt.legend()
plt.show()


## 7) Summary table

In [ ]:
rows = []
for k in ['diabetes','heart','cancer']:
    tm = metrics[k]['test_metrics']
    rows.append({
        'task': k,
        'dataset': metrics[k]['dataset'],
        'rows': metrics[k]['n_rows'],
        'best_model': metrics[k]['best_model'],
        **tm,
    })
summary = pd.DataFrame(rows)
display(summary)
